# CliMaPan-Lab Quick Start

Run a minimal economic simulation and inspect the results using AMBER's DataFrame-backed agent system.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from climapan_lab.base_params import economic_params
from climapan_lab.src.models import EconModel
import polars as pl
import numpy as np

## 1. Configure a small simulation

In [ ]:
params = economic_params.copy()
params.update({
    'c_agents': 100,           # 100 consumers
    'capitalists': 10,         # 10 firm owners
    'csf_agents': 3,           # 3 consumer-goods firms
    'cpf_agents': 2,           # 2 capital-goods firms
    'steps': 60,               # ~2 months of daily steps
    'seed': 42,
    'show_progress': False,
    'covid_settings': None,    # no pandemic
    'climateModuleFlag': False,
    'verboseFlag': False,
})

print(f'Agents: {params["c_agents"]} consumers + firms')
print(f'Steps: {params["steps"]} days')

## 2. Run the model

In [ ]:
model = EconModel(params)
result = model.run()

print(f'Completed in {result["info"]["run_time"]:.2f}s')
print(f'Steps: {result["info"]["steps"]}')
print(f'Agent rows: {result["agents"].height}')
print(f'Model columns: {result["model"].columns}')

## 3. Inspect the agent population

AMBER stores all agents as a Polars DataFrame. You can query it directly.

In [ ]:
agents_df = result['agents']
print(f'Shape: {agents_df.shape}')
print(f'Columns: {agents_df.columns}')
agents_df.head(5)

### Filter agents by type (consumerType column)

In [ ]:
if 'consumerType' in agents_df.columns:
    workers = agents_df.filter(pl.col('consumerType') == 'workers')
    capitalists = agents_df.filter(pl.col('consumerType') == 'capitalists')
    print(f'Workers: {workers.height}')
    print(f'Capitalists: {capitalists.height}')
    print(f'Avg worker wage: {workers["wage"].mean():.2f}')
    print(f'Avg capitalist deposit: {capitalists["deposit"].mean():.2f}')

## 4. Model-level metrics

The model records GDP, unemployment, and more each month.

In [ ]:
model_df = result['model']
key_cols = [c for c in ['GDP', 'UnemploymentRate', 'Gini', 'Investment'] if c in model_df.columns]
model_df.select(key_cols).tail(5)

## 5. Using AMBER's view API on live agents

During simulation, you can query agents with `model.agents.where(...)`

In [ ]:
# Re-run a single step and inspect the live agent list
p = economic_params.copy()
p.update({'steps': 5, 'c_agents': 20, 'seed': 42, 'show_progress': False,
          'covid_settings': None, 'climateModuleFlag': False})

m = EconModel(p)
m.setup()

# Access agent lists directly
print(f'Consumer agents: {len(m.consumer_agents)}')
print(f'CS firms: {len(m.csfirm_agents)}')
print(f'CP firms: {len(m.cpfirm_agents)}')

# Use AMBER view API to query columns
if len(m.consumer_agents) > 0:
    wages = m.consumer_agents.getWage()
    print(f'Wages (first 5): {wages[:5]}')
    print(f'Mean wage: {np.mean(wages):.2f}')